# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para las celdas Markdown.

Material educativo sujeto a revisión y validación en Microsoft Fabric.

# NB_08 — AI_EMBED_PANDAS
## 1. Objetivos y requisitos
Generar embeddings por fila, validar dimensionalidad, calcular similitud coseno y auditar tokens. Usar Microsoft Fabric con capacidad y AI Functions habilitadas, runtime compatible (1.3 o posterior; revisar configuración específica del runtime Python). No se necesita una API externa. Los datos son ficticios. Ejecutar inferencias genera consumo real.

In [1]:
# Instalación opcional en runtime Python; ejecutar una vez si faltan las bibliotecas.
INSTALL = False
if INSTALL:
    import subprocess, sys
    from urllib.request import urlretrieve
    wheels = []
    for url, name in [
        ("https://aka.ms/fabric-aifunctions-whl", "synapseml_internal-latest-py3-none-any.whl"),
        ("https://aka.ms/fabric-synapseml-core-whl", "synapseml_core-latest-py3-none-any.whl")
    ]:
        urlretrieve(url, name)
        wheels.append(name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *wheels])
# Reiniciar la sesión si el runtime lo requiere.

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 3, Finished, Available, Finished, False)

## 2. Configuración
Se utiliza el endpoint integrado de Fabric en una sesión limpia. `embedding_deployment_name` selecciona el modelo vectorial, no `model_deployment_name`. Ada produce 1536 componentes. Concurrencia de laboratorio: 10 (predeterminada: 200). La barra de progreso muestra métricas. No se admiten embeddings mediante el endpoint Foundry personalizado descrito en la documentación de configuración.

In [2]:
import pandas as pd
import numpy as np
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from time import perf_counter
import uuid
MODEL_NAME = "text-embedding-ada-002"
EXPECTED_DIM = 1536
CONCURRENCY = 10
aifunc.default_conf.embedding_deployment_name = MODEL_NAME
aifunc.default_conf.concurrency = CONCURRENCY
aifunc.default_conf.progress_bar_mode = "stats"

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 4, Finished, Available, Finished, False)

## 3. Referencia oficial del método
[Microsoft Learn — ai.embed con Pandas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/embed)

```python
df["embed"] = df["col1"].ai.embed()
```

| Parámetro | Tipo | Descripción |
|---|---|---|
| Ninguno obligatorio | — | Se invoca sobre una Series de textos. |
| `conf` | Conf | Configuración opcional por llamada, documentada en la página de configuración. |

Retorna una Series de arrays NumPy float32; su dimensión depende del modelo. Ejemplo de configuración individual: `df["descripcion"].ai.embed(conf=aifunc.Conf(embedding_deployment_name=MODEL_NAME))`. No ejecutar ambos ejemplos salvo que se desee repetir la inferencia.

## 4. Dataset ficticio
Se reutilizan las seis solicitudes de soporte del NB_08 PySpark para facilitar la comparación.

In [3]:
data = [(1, "No puedo acceder al portal corporativo"),
        (2, "El inicio de sesión en el portal falla"),
        (3, "La carga nocturna de ventas no terminó"),
        (4, "El pipeline de ventas falló durante la madrugada"),
        (5, "Solicito una silla ergonómica para la oficina"),
        (6, "Necesito restablecer mi contraseña")]
df = pd.DataFrame(data, columns=["id_ticket", "descripcion"])
display(df)

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1d13699c-e3d8-421c-aa72-78dce3380549)

## 5. Generación de embeddings
Una sola llamada; conservar el resultado original y sus estadísticas antes de transformar los vectores. El tiempo corresponde a esta llamada y la memoria de entrada es una aproximación de Pandas.

In [4]:
run_id = str(uuid.uuid4())
started = datetime.now(timezone.utc).isoformat()
input_bytes = int(df.memory_usage(deep=True).sum())
t0 = perf_counter()
embeddings = df["descripcion"].ai.embed()
elapsed = perf_counter() - t0
raw = embeddings.ai.stats
resultado = df.assign(embedding=embeddings)
print("Tiempo (s):", elapsed)

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 6, Finished, Available, Finished, False)

ai.embed:  83%|████████▎ | 5/6 [00:33<00:03,  3.89s/it, cached=0->0.00, in=52->62.40, out=0->0.00, CU-s=0.17->0.21]

Tiempo (s): 37.99534045100006


## 6. Validación de vectores
Verificar tipo, dimensión, valores finitos y norma positiva. No mostrar los 1536 componentes completos. Un valor ausente no permite determinar por sí solo la causa de error.

In [5]:
def validar(v):
    if v is None or v is pd.NA or (isinstance(v, float) and np.isnan(v)):
        return "MISSING"
    if not isinstance(v, np.ndarray):
        return "INVALID_TYPE"
    if v.shape != (EXPECTED_DIM,):
        return "INVALID_DIM"
    if v.dtype != np.float32:
        return "INVALID_DTYPE"
    if not np.isfinite(v).all() or np.linalg.norm(v.astype("float64")) == 0:
        return "INVALID_VECTOR"
    return "OK"
resultado["validation_status"] = embeddings.map(validar)
resultado["dimension"] = embeddings.map(lambda v: v.size if isinstance(v, np.ndarray) else None)
resultado["preview"] = embeddings.map(lambda v: v[:5].tolist() if isinstance(v, np.ndarray) else None)
display(resultado.drop(columns="embedding"))
display(resultado.validation_status.value_counts(dropna=False))

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bef7f206-eb2d-4f18-8602-f247a7fa91c4)

SynapseWidget(Synapse.DataFrame, 3a2b8a48-de20-431c-8aa8-94c65fc629c6)

## 7. Estadísticas de ejecución
Consultar métricas de la Series original. Los campos ausentes se mantienen como nulos, no se inventan tokens.

In [6]:
def normalizar(raw):
    if isinstance(raw, pd.DataFrame) and len(raw)==1:
        return raw.iloc[0].to_dict()
    if isinstance(raw, pd.Series):
        return raw.to_dict()
    if isinstance(raw, dict):
        return raw
    raise RuntimeError("Revisar estructura de ai.stats")

display(raw)
stats = normalizar(raw)


StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c31b51d1-db0b-40cc-8ba6-0ad7c371d31e)

## 8. Consumo de capacidad
Tarifa publicada para Ada: 3,36 CU-segundos por 1000 tokens de entrada (ventana de contexto 8192 tokens). No aplicar tarifas GPT-5-mini ni sumar tokens de salida o razonamiento. Ejemplo hipotético: 1000 tokens = 3,36 CU-s. Estimación válida únicamente para el modelo y endpoint integrados identificados. No incluye cómputo del notebook ni almacenamiento. Verificar tarifas antes de presupuestar.

[Facturación oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

In [7]:
def metric(k):
    v = stats.get(k)
    return None if v is None or pd.isna(v) else int(v)
I = metric("input_tokens")
cu = None
if stats.get("model") == MODEL_NAME and stats.get("client_type") == "fabric_llm_endpoint" and I is not None and I >= 0:
    cu = I * 3.36 / 1000
print("Input tokens:", I, "CU-s estimados:", cu)
print("CU-h estimadas:", None if cu is None else cu / 3600)

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 9, Finished, Available, Finished, False)

Input tokens: 61 CU-s estimados: 0.20495999999999998
CU-h estimadas: 5.6933333333333326e-05


## 9. Auditoría
Mantener columnas compatibles con la serie Pandas. La memoria reportada de salida incluye el tamaño de la Series y los arrays; no equivale a la memoria máxima del proceso.

In [8]:
audit = dict(run_id=run_id, execution_timestamp_utc=started, notebook_name="NB_08_AI_EMBED_PANDAS", operation="ai.embed", model_configured=MODEL_NAME, model_reported=stats.get("model"), num_input_rows=len(df), client_type=stats.get("client_type"), estimated_cu_seconds=cu, elapsed_seconds=elapsed, input_bytes=input_bytes, output_bytes=int(embeddings.memory_usage(deep=True)), concurrency=CONCURRENCY)
for key in ("num_successful","num_exceptions","num_unevaluated","num_harmful","input_tokens","cached_tokens","output_tokens","reasoning_tokens"):
    audit[key] = metric(key)
if (audit["num_exceptions"] or 0)>0 or (audit["num_harmful"] or 0)>0:
    audit["execution_status"] = "PARTIAL"
elif audit["num_successful"] == len(df) and audit["num_exceptions"] == 0 and audit["num_harmful"] == 0 and resultado.validation_status.eq("OK").all():
    audit["execution_status"] = "SUCCESS"
else:
    audit["execution_status"] = "INCOMPLETE"
audit_df = pd.DataFrame([audit])
display(audit_df)

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e198cf83-b14e-4c31-9c7e-f9afbadc3cfd)

## 10. Persistencia opcional de auditoría
Requiere Lakehouse adjunto, permisos y paquetes deltalake y pyarrow. Ruta bajo Files, no tabla SQL registrada. Reejecutar la escritura agrega registros duplicados.

In [9]:
SAVE_DELTA = False
if SAVE_DELTA:
    from deltalake import write_deltalake
    import pyarrow as pa
    schema = pa.schema([(k, pa.int64() if k.startswith("num_") or k.endswith("_tokens") or k in ("input_bytes", "output_bytes", "concurrency") else pa.float64() if k in ("estimated_cu_seconds", "elapsed_seconds") else pa.string()) for k in audit_df.columns])
    write_deltalake("/lakehouse/default/Files/audit_ai_functions_pandas", pa.Table.from_pandas(audit_df, schema=schema, preserve_index=False), mode="append")

StatementMeta(, b1323cb5-99e9-480b-b9d3-348630680c97, 11, Finished, Available, Finished, False)

## 11. Similitud coseno local con NumPy
Comparar el ticket 1 con los demás vectores válidos. La similitud es el producto escalar dividido por el producto de las normas. Rango matemático [-1, 1]. No es una probabilidad. No se realizan nuevas inferencias; solo cálculo local. Si falta el vector de referencia, se omite el ranking.

In [ ]:
valid = resultado.loc[resultado.validation_status.eq("OK")].copy()
ref = valid.loc[valid.id_ticket.eq(1), "embedding"]
if len(ref) == 1:
    a = ref.iloc[0].astype("float64")
    def cosine(v):
        b = v.astype("float64")
        return float(np.clip(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)), -1, 1))
    ranking = valid.loc[valid.id_ticket.ne(1), ["id_ticket","descripcion","embedding"]].copy()
    ranking["cosine_similarity"] = ranking.embedding.map(cosine)
    display(ranking.drop(columns="embedding").sort_values("cosine_similarity", ascending=False))
else:
    print("Referencia no disponible; ranking omitido.")

## 12. Persistencia opcional de vectores
Guardar exclusivamente vectores válidos como listas float32 con metadata del modelo y run_id. No mezclar espacios vectoriales de distintos modelos. La ruta es independiente de la auditoría. Revisar privacidad y permisos antes de persistir textos o embeddings.

In [ ]:
RUN_SAVE_VECTORS = False
if RUN_SAVE_VECTORS and not valid.empty:
    import pyarrow as pa
    from deltalake import write_deltalake
    table = pa.table({"id_ticket": pa.array(valid.id_ticket, type=pa.int64()), "descripcion": pa.array(valid.descripcion, type=pa.string()), "embedding_array": pa.array([v.tolist() for v in valid.embedding], type=pa.list_(pa.float32())), "dimension": pa.array([EXPECTED_DIM]*len(valid), type=pa.int64()), "embedding_model": [MODEL_NAME]*len(valid), "run_id": [run_id]*len(valid)})
    write_deltalake("/lakehouse/default/Files/demo_ticket_embeddings_pandas", table, mode="append")

## 13. Comparación y conclusiones
PySpark: `df.ai.embed(input_col="descripcion", output_col="embedding", error_col="error_embedding")` con procesamiento distribuido. Pandas: `df["descripcion"].ai.embed()` con almacenamiento local y solicitudes asíncronas. Ambos pueden generar consumo del endpoint integrado. La comparación real exige ejecutar en condiciones equivalentes y revisar Capacity Metrics. No se han ejecutado inferencias al crear este notebook.

Referencias: [Método](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/embed) · [Configuración](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration) · [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing).